# H1 — Thinking-Budget Study on LFM2.5-2.6B

**Kaggle 2×T4, data-parallel, self-contained.** Everything needed to run the full experiment is
in this notebook; Kaggle provides 30 GPU-hours/week on 2× T4 with no billing setup.

## The question

This is not simply "does accuracy rise when the model thinks more". It is a **calibration**
study:

> Does LFM2.5-2.6B spend reasoning tokens where they actually buy accuracy, or does it
> **overthink** (burn budget on problems already solved at a small budget) and **underthink**
> (run out of budget before it could have answered correctly)?

Two sub-questions follow:

1. **Quantity** — at which budget does each benchmark's accuracy curve flatten (the *knee*), and
   does that knee move with difficulty the way it should?
2. **Quality** — does a longer trace reason *better*, or does it loop and thrash?

The independent variable is a **hard cap on thinking tokens**, not a prompt asking for brevity.
A decoding-time cap changes only the compute; prompt-level length control would confound
reasoning with instruction-following.

## The grid

```
9 benchmarks × 10 questions × 7 compute levels × 5 samples ≈ 3,150 calls
COMPUTE_LEVELS = [0, 256, 512, 1024, 2048, 4096, 8192]
```

- **Log-spaced budgets.** Accuracy-vs-compute is roughly logarithmic, so linear spacing would
  waste most points in the flat region. 256 ≈ "no room to think" (a proxy for raw difficulty);
  8192 is the practical ceiling under this context length.
- **5 samples per (question, budget)** at `temperature=0.7`. One sample is a coin flip on a hard
  item; five give a per-question accuracy *and* a stability measure. At temperature 0 the five
  samples would be near-identical and variance unmeasurable.
- **9 benchmarks in 3 tiers.** The hypothesis is about the *relationship* between difficulty and
  optimal budget, which needs at least an ordinal difficulty axis.

## Model

LFM2.5-2.6B (LiquidAI) is a 2.6B *pure-reasoning* hybrid — 22 gated short-conv blocks + 8 GQA
blocks, 128K context. Its chat template always opens a ` thinking` segment and switches to the
answer with ` response`. That explicit boundary is what makes a thinking budget enforceable and
the two halves separable afterwards. At ~5.2 GB in fp16 it also fits a single T4 entirely.

## Running it

1. **Kaggle → Code → New Notebook**, then *File → Import/Upload Notebook*.
2. **Settings → Accelerator:** `GPU T4x2`. **Internet ON.**
3. **(GPQA only)** Settings → **Secrets** → add `HF_TOKEN`, and accept the terms at
   <https://huggingface.co/datasets/Idavidrein/gpqa> with that account.
4. **Run all.** Install ≈5 min (first time), servers ≈2–5 min, pilot <1 min, full sweep
   ≈20–30 min on 2×T4 (≈40 min on 1×T4).
5. **Results:** Output tab → *Download All* — `h1_raw_results.csv`, `h1_pilot_results.csv`,
   `h1_analysis_outputs/`.

If the session drops, just **Run all** again: the sweep checkpoints every 25 calls and resumes.

## 1. Environment

Three things are installed here, each for a specific reason:

- **`vllm-thinking-budget` fork** — stock vLLM has no per-request thinking cap. The fork adds a
  logits processor that watches the generated tokens and, at `budget − 1`, forces the ` response`
  token so the model *must* stop thinking and answer. It runs against a stock `vllm` wheel via
  `PYTHONPATH`, so no source build is needed.
- **`torchaudio` removal** — the Kaggle base image bundles a build that conflicts with the torch
  version the vLLM wheel pins. It is unused here.
- **Sparse clone of `gorilla`** — BFCL data only. A full clone is large and slow; the sparse
  checkout pulls just the `bfcl_eval/data` subtree.

`HF_TOKEN` is optional and only needed for the gated GPQA dataset.

PLEASE Restart the kernel/runtime after installing requirements.

In [ ]:
# Thinking-budget vLLM fork (works with stock `vllm` >=0.17 via PYTHONPATH).
!git clone --depth 1 https://github.com/phishingupstream/vllm-thinking-budget.git

# Runtime deps. This pulls/upgrades torch to match the vLLM wheel.
!pip install -q -U vllm datasets openai "pandas<3" "numpy==2.3.5" matplotlib "kneed"

# Kaggle base image bundles torchaudio which conflicts with the pinned torch —
# unused here, so remove it.
!pip uninstall -y -q torchaudio

# BFCL v4 data: sparse clone of ONLY the bfcl_eval/data subtree (fast, small).
!git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/ShishirPatil/gorilla.git /kaggle/working/gorilla
!git -C /kaggle/working/gorilla sparse-checkout set \
    berkeley-function-call-leaderboard/bfcl_eval/data

# HF login for the gated GPQA dataset.
import os
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    print("[hf] HF_TOKEN loaded from Kaggle secrets")
except Exception as e:
    print(f"[hf] no HF_TOKEN secret: {e}")

import vllm, datasets, openai, pandas
print(f"vllm={vllm.__version__} datasets={datasets.__version__}")

# Sanity: the fork's processor file must be importable.
print("fork present:",
      os.path.isdir("/kaggle/working/vllm-thinking-budget"),
      os.path.exists("/kaggle/working/vllm-thinking-budget/thinking_budget_processor.py"))
print("gorilla BFCL data present:",
      os.path.exists("/kaggle/working/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v4_simple_python.json"))

# PLEASE Restart the kernel after installing requirements.

fatal: destination path 'vllm-thinking-budget' already exists and is not an empty directory.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.7 MB/s eta 0:00:0000:01
fatal: destination path '/kaggle/working/gorilla' already exists and is not an empty directory.
[hf] HF_TOKEN loaded from Kaggle secrets
vllm=0.29.0 datasets=5.0.1
fork present: True True
gorilla BFCL data present: True


## 2. Serving layer — one vLLM server per GPU

### Why data parallelism, not tensor parallelism

A 2.6B model fits entirely on one 16 GB T4. Splitting it with TP=2 would add an all-reduce per
layer across two GPUs — pure communication overhead for a memory problem that does not exist.
The right way to use N GPUs here is **data parallelism**: one independent vLLM instance per GPU
(TP=1 each), with the evaluator round-robining requests across `server.ports`. That gives ≈2×
throughput instead of ≈1× with extra latency. With a single T4 the same code runs unchanged on
one server.

### Why the fork needs patching

Under vLLM's async scheduling, the token buffer the thinking-budget processor scans can contain
`-1` placeholders for positions that are not committed yet. The unpatched processor treats `-1`
as a real token id and miscounts how much thinking has happened — so the budget stops binding
correctly. `apply_patch()` makes the scan stop at the first `-1`.

### Server configuration

`enable-prefix-caching` matters more than usual here: every question is sent 6 budgets × 5
samples = 30 times, so caching the prompt prefill makes the sweep decode-bound rather than
prefill-bound. `max-model-len` and `gpu-memory-utilization` step down automatically if VRAM per
GPU comes back low, so the notebook degrades instead of OOM-ing.

In [37]:
import os
import json
import time
import math
import random
import glob
import ast
import re
import subprocess
import tempfile
import urllib.request
import urllib.error

REPO_DIR = "/kaggle/working/vllm-thinking-budget"
WORK_DIR = "/kaggle/working"


def detect_gpus():
    """Return (gpu_names:list, gpu_count). Also exposes GPU_RATE_USD_S later."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip().splitlines()
        names = [l.rsplit(",", 1)[0].strip() for l in out]
        return names, len(names)
    except Exception:
        return [], 0


class VLLMServer:
    """Launches one vLLM server per GPU (TP=1 each) and exposes their ports.

    ``self.ports`` is the list the evaluator round-robins over.
    """

    def __init__(self, model, port=8001, n_servers=1, repo_dir=REPO_DIR,
                 work_dir=WORK_DIR):
        self.model = model
        self.port = port  # first server port
        self.n_servers = max(1, n_servers)
        self.ports = [port + i for i in range(self.n_servers)]
        self.repo_dir = repo_dir
        self.work_dir = work_dir
        self.processes = []
        self.gpu_names, self.gpu_count = [], 0
        self.tp_size = 1  # TP=1 per GPU; DP handled by multiple servers

    # ---- fork patch: stop the token scan at the first -1 placeholder --------
    def apply_patch(self):
        path = os.path.join(self.repo_dir, "thinking_budget_processor.py")
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        content = open(path).read()
        old = ("        for i in range(self.last_scanned, end):\n"
               "            tid = tokens[i]")
        new = ("        for i in range(self.last_scanned, end):\n"
               "            tid = tokens[i]\n"
               "            if tid == -1:\n"
               "                end = i\n"
               "                break")
        if "if tid == -1:" in content:
            print("[server] \u2713 patch already applied")
        elif old in content:
            open(path, "w").write(content.replace(old, new))
            print("[server] \u2713 patch applied")
        else:
            print("[server] \u26a0 patch pattern not found \u2014 continuing anyway")

    # ---- config ------------------------------------------------------------
    def write_config(self):
        # Robust VRAM detection: sum all GPUs, handle MiB suffix / blanks
        gpu_count, total_vram_gb, gpu_names = 0, 0, []
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,memory.total",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=10,
            ).stdout.strip()
            for line in out.splitlines():
                line = line.strip()
                if not line:
                    continue
                parts = [p.strip() for p in line.rsplit(",", 1)]
                if len(parts) == 2:
                    name, mem = parts
                    gpu_names.append(name)
                    total_vram_gb += int(float(mem)) / 1024.0
                    gpu_count += 1
        except Exception as e:
            print(f"[server] nvidia-smi failed: {e}", flush=True)
        self.gpu_count = gpu_count
        self.gpu_names = gpu_names
        usable = max(1, min(self.n_servers, gpu_count))
        self.ports = self.ports[:usable]  # never launch more servers than GPUs
        print(f"[server] detected {gpu_count} GPU(s): {gpu_names} "
              f"total={total_vram_gb:.0f}GB \u2192 {usable} server(s), TP=1 each",
              flush=True)

        # LFM2.5-2.6B is a 2.6B dense hybrid (22 conv + 8 GQA blocks): ~5.2 GB
        # in fp16, so one T4 (16 GB) holds it with room for a big KV cache.
        per_gpu_gb = total_vram_gb / max(1, usable)
        max_model_len, util, max_num_seqs = 32768, 0.95, 128
        if per_gpu_gb > 0 and per_gpu_gb < 12000:
            max_model_len, util, max_num_seqs = 16384, 0.92, 64
            print("[server] \u26a0 low VRAM per GPU \u2014 max-model-len=16384", flush=True)

        self.config_paths = []
        for i, port in enumerate(self.ports):
            name = "vllm-config.yaml" if i == 0 else f"vllm-config-{port}.yaml"
            path = os.path.join(self.work_dir, name)
            open(path, "w").write(
                f"model: {self.model}\n"
                f"max-model-len: {max_model_len}\n"
                f"tensor-parallel-size: 1\n"
                f"gpu-memory-utilization: {util}\n"
                f"max-num-seqs: {max_num_seqs}\n"
                "enable-prefix-caching: true\n"
                f"port: {port}\n"
                "logits-processors:\n"
                '- "thinking_budget_processor:ThinkingBudgetLogitsProcessor"\n'
            )
            self.config_paths.append(path)
        self.config_path = self.config_paths[0]
        print(f"[server] \u2713 wrote {len(self.config_paths)} config(s): "
              f"{[os.path.basename(p) for p in self.config_paths]}")
        print(f"[server] tp_size={self.tp_size} max-model-len={max_model_len} "
              f"max-num-seqs={max_num_seqs} util={util}")

    # ---- lifecycle ---------------------------------------------------------
    def _wait_ready(self, proc, port, log_path, timeout=600):
        start_time = time.time()
        last_report = 0
        while time.time() - start_time < timeout:
            if proc.poll() is not None:
                print(f"[server] \u274c vLLM on port {port} exited rc={proc.returncode} "
                      "\u2014 last 30 log lines:", flush=True)
                for line in open(log_path).readlines()[-30:]:
                    print("   " + line.rstrip(), flush=True)
                raise RuntimeError(
                    f"vLLM on port {port} exited rc={proc.returncode}")
            try:
                r = urllib.request.urlopen(
                    f"http://localhost:{port}/health", timeout=2)
                if r.status == 200:
                    print(f"[server] \u2713 port {port} READY after "
                          f"{time.time()-start_time:.0f}s", flush=True)
                    return
            except (urllib.error.URLError, urllib.error.HTTPError):
                pass
            elapsed = time.time() - start_time
            if elapsed - last_report >= 15:
                print(f"[server] port {port} still starting... "
                      f"{elapsed:.0f}s/{timeout}s", flush=True)
                last_report = elapsed
            time.sleep(2)
        raise RuntimeError(
            f"vLLM on port {port} not healthy in time \u2014 check {log_path}")

    def start(self, timeout=600):
        print("\n" + "=" * 60, flush=True)
        print(f"[server] GPUs: {self.gpu_names or 'none detected'} "
              f"(count={self.gpu_count}) servers={len(self.ports)}", flush=True)
        print("[server] starting vLLM (one per GPU, TP=1 each)...", flush=True)
        print("=" * 60, flush=True)

        subprocess.run(["pkill", "-9", "-f", "vllm"], check=False)
        subprocess.run(["pkill", "-9", "-f", "VLLM::"], check=False)
        time.sleep(3)

        env = os.environ.copy()
        env["PYTHONPATH"] = self.repo_dir + ":" + env.get("PYTHONPATH", "")

        self.processes = []
        for i, port in enumerate(self.ports):
            proc_env = env.copy()
            proc_env["CUDA_VISIBLE_DEVICES"] = str(i)  # one GPU per server
            log_path = os.path.join(self.work_dir, f"vllm-{port}.log")
            log_file = open(log_path, "w")
            p = subprocess.Popen(
                ["vllm", "serve", self.model, "--config", self.config_paths[i]],
                env=proc_env, stdout=log_file, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            self.processes.append(p)
            print(f"[server] \u2713 port {port} started pid={p.pid} "
                  f"(CUDA_VISIBLE_DEVICES={i})", flush=True)
            self._wait_ready(p, port, log_path, timeout=timeout)
        print(f"[server] \u2713 ALL SERVERS READY: {self.ports} \u2014 "
              f"http://localhost:{self.ports[0]}/v1 ...", flush=True)

    def stop(self):
        subprocess.run(["pkill", "-9", "-f", "vllm"], check=False)
        subprocess.run(["pkill", "-9", "-f", "VLLM::"], check=False)
        print("[server] vLLM stopped.")

    def setup_and_start(self):
        self.apply_patch()
        self.write_config()
        self.start()

## 3. Launch and pre-flight check

### Why the tokenizer check must run before anything else

The budget processor resolves the boundary tags **by name** —
`convert_tokens_to_ids(" thinking")`, `(" response")`, plus the newline token it forces at
`budget − 1`. If any of those does not encode to exactly **one** token in this tokenizer, the
processor can never force the switch, and every single call silently burns `max_tokens` instead
of respecting its budget.

That failure mode is dangerous precisely because it is invisible: the run completes, the CSV
fills up, the curves look plausible, and none of the numbers mean anything. So the check raises
and aborts rather than warning. For LFM2.5 the tags are dedicated single tokens (ids 124901 /
124902).

`_tok` loaded here is reused downstream by the evaluator to count thinking vs. output tokens, so
no second tokenizer download is needed.

In [38]:
Model = "LiquidAI/LFM2.5-2.6B"

# One server per detected GPU; the evaluator round-robins over server.ports.
_gpu_names, _gpu_count = detect_gpus()
N_SERVERS = _gpu_count if _gpu_count >= 1 else 1

server = VLLMServer(model=Model, port=8001, n_servers=N_SERVERS)
server.setup_and_start()

# Billing/throughput knobs for live KPIs (T4 $0.164/hr = $0.000164/s).
GPU_COUNT = len(server.ports)
GPU_RATE_USD_S = 0.000164 * GPU_COUNT
MAX_WORKERS = 16 * GPU_COUNT          # 16 concurrent requests per server
print(f"[run] gpus={GPU_COUNT} rate=${GPU_RATE_USD_S*3600:.2f}/hr "
      f"workers={MAX_WORKERS}")



# Pre-flight: the budget processor resolves the tags by name, so each must
# encode to exactly one token. Abort loudly if not.

from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(Model, trust_remote_code=True)
THINKING_TOKEN_ID = 124901
RESPONSE_TOKEN_ID = 124902
THINKING_MARKER = _tok.decode([THINKING_TOKEN_ID])
RESPONSE_MARKER = _tok.decode([RESPONSE_TOKEN_ID])

for _name, _tid, _marker in [
    ("thinking", THINKING_TOKEN_ID, THINKING_MARKER),
    ("response", RESPONSE_TOKEN_ID, RESPONSE_MARKER),
]:
    _enc = _tok.encode(_marker, add_special_tokens=False)
    _ok = len(_enc) == 1 and _enc[0] == _tid
    print(f"[run] tok {_name}: id={_tid} decoded={_marker!r} round_trip={_enc} ok={_ok}")
    if not _ok:
        raise RuntimeError(
            f"LFM2.5 token id {_tid} ({_name}) does not round-trip through "
            f"decode->encode as a single token ({_enc}); the thinking-budget "
            "processor and the text split would not agree - aborting.")

_nl_enc = _tok.encode("\n", add_special_tokens=False)
_nl_ok = len(_nl_enc) == 1 and _nl_enc[0] not in (None, -1)
print(f"[run] tok '\\n': id={_nl_enc[0] if _nl_ok else _nl_enc} ok={_nl_ok}")
if not _nl_ok:
    raise RuntimeError(f"'\\n' is not a single token ({_nl_enc}) - aborting.")

print(f"[run] tokenizer OK: thinking={THINKING_MARKER!r} response={RESPONSE_MARKER!r}")

[server] ✓ patch already applied
[server] detected 2 GPU(s): ['Tesla T4', 'Tesla T4'] total=30GB → 2 server(s), TP=1 each
[server] ⚠ low VRAM per GPU — max-model-len=16384
[server] ✓ wrote 2 config(s): ['vllm-config.yaml', 'vllm-config-8002.yaml']
[server] tp_size=1 max-model-len=16384 max-num-seqs=64 util=0.92

[server] GPUs: ['Tesla T4', 'Tesla T4'] (count=2) servers=2
[server] starting vLLM (one per GPU, TP=1 each)...
[server] ✓ port 8001 started pid=2183 (CUDA_VISIBLE_DEVICES=0)
[server] port 8001 still starting... 16s/600s
[server] port 8001 still starting... 32s/600s
[server] port 8001 still starting... 48s/600s
[server] port 8001 still starting... 64s/600s
[server] ✓ port 8001 READY after 76s
[server] ✓ port 8002 started pid=2339 (CUDA_VISIBLE_DEVICES=1)
[server] port 8002 still starting... 16s/600s
[server] port 8002 still starting... 32s/600s
[server] port 8002 still starting... 48s/600s
[server] port 8002 still starting... 64s/600s
[server] ✓ port 8002 READY after 70s
[server

## 4. Benchmark suite

Nine benchmarks across three assumed difficulty tiers and six task types:

| Benchmark | Tier | Task type | Source |
|---|---|---|---|
| GSM8K | easy | math | `openai/gsm8k` |
| MMLU-Redux | easy | general knowledge | `edinburgh-dawg/mmlu-redux-2.0` (4 subjects) |
| IFEval | easy | instruction following | `google/IFEval` |
| MATH-full | medium | math | `DigitalLearningGmbH/MATH-lighteval` |
| MMLU-Pro | medium | general knowledge | `TIGER-Lab/MMLU-Pro` |
| BFCL-v4 simple_python | medium | agentic tool use | sparse clone of `ShishirPatil/gorilla` |
| AIME2025 | hard | math | `test-time-compute/aime_2025` |
| GPQA-Diamond | hard | PhD science | `Idavidrein/gpqa` (gated) |
| LiveCodeBench-v6 | hard | coding | `lighteval/code_generation_lite` |

### Difficulty tiers are hypotheses, not facts

Liquid AI publishes official LFM2.5-2.6B numbers for only two of these (AIME25 = 51.87,
LiveCodeBench-v6 = 59.41). For the rest, the easy/medium/hard label is a literature-based prior.
Because every tier-level result downstream inherits these labels, the analysis cell re-checks
them empirically against this model's own low-budget accuracy and flags mismatches.

### Every grader takes the *last* match

A reasoning model writes "maybe A… no, B fits better". Taking the **first** regex match returns
A — the scratch mention — instead of B, the model's actual pick. Every grader here therefore
scans from the end: last standalone letter, last number, last `\boxed{}`, last fenced code block,
last parseable function call. The evaluator already strips the thinking text upstream, so this is
defensive rather than load-bearing, but it costs nothing and removes a whole class of silent
mis-grading.

### Deliberate simplifications

Three graders are lighter than the official protocols, and each is documented in its class:

- **IFEval** — implements ~8 of the ~25 official verifiable-instruction checkers and keeps only
  questions that use them. Unsupported instruction ids are skipped rather than failed.
- **BFCL** — syntactic call matching instead of the official semantic AST checker.
- **LiveCodeBench** — grades on **public** test cases only, so scores are an optimistic bound.

These are acceptable for a *within-model, across-budget* comparison: every budget level passes
through the same lenient grader, so the *shape* of the curve survives even though the *level* is
biased. They are not comparable to published leaderboard numbers.

In [43]:
from abc import ABC, abstractmethod
from datasets import load_dataset

import subprocess
import tempfile
import os
import re
import ast
import json
import glob
import random
import math
import zlib
import base64


class Benchmark(ABC):
    """Base class — implement load() and grade() for any new benchmark.

    Every subclass declares:
      - name: str, unique identifier used in the raw results table
      - difficulty_tier: "easy" | "medium" | "hard" — a working hypothesis
        from the literature difficulty ordering, not a verified fact about
        LFM2.5-2.6B; the analysis cell re-checks it empirically.
      - task_type: category label used for disaggregation (matches the target
        benchmark table: math / general_knowledge / instruction_following /
        agentic_tool_use / phd_science / coding)
    """

    name = None
    difficulty_tier = None
    task_type = None

    @abstractmethod
    def load(self, n):
        """Returns list of {"question": str, "gold": any}"""
        ...

    @abstractmethod
    def grade(self, output_text, gold):
        """Returns True/False. `output_text` is the final-answer portion only;
        the evaluator strips the thinking text upstream.
        """
        ...


def _extract_boxed(text):
    """Extract the last \\boxed{...} contents, handling nested braces."""
    idxs = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not idxs:
        return None
    start = idxs[-1] + len("\\boxed{")
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start:i - 1].strip()


# ================================================================== easy tier
class GSM8KBenchmark(Benchmark):
    """Easy / Math — GSM8K (grade-school arithmetic word problems).

    Tier: "easy" is a literature-based hypothesis — GSM8K sits near ceiling for
    modern ~2-8B reasoning models. Liquid AI publishes no LFM2.5-2.6B GSM8K
    score, so the tier is validated empirically in the analysis cell.
    """

    name = "GSM8K"
    difficulty_tier = "easy"
    task_type = "math"

    def load(self, n=30):
        ds = load_dataset(
            "openai/gsm8k",
            "main",
            split="test").select(
            range(n))
        questions = []
        for row in ds:
            gold = re.search(
                r"####\s*(-?[\d,]+\.?\d*)",
                row["answer"]).group(1).replace(
                ",",
                "")
            questions.append({"question": row["question"], "gold": gold})
        return questions

    def grade(self, output_text, gold):
        numbers = re.findall(r"-?\d[\d,]*\.?\d*", output_text)
        if not numbers:
            return False
        try:
            return abs(
                float(numbers[-1].replace(",", "")) - float(gold)) < 1e-4
        except ValueError:
            return False


class MMLURedux(Benchmark):
    """Easy / General knowledge — MMLU-Redux.

    MMLU-Redux re-annotates a subset of the MMLU test sets to fix labeling
    errors, so it shares MMLU's per-subject schema; a handful of subjects are
    mixed here for a representative sample.

    Tier: "easy" is a literature-based hypothesis — the relabeling keeps MMLU's
    difficulty profile, well below PhD-level suites like GPQA. No official
    LFM2.5-2.6B score exists.
    """

    name = "MMLU-Redux"
    difficulty_tier = "easy"
    task_type = "general_knowledge"

    DEFAULT_SUBJECTS = [
        "high_school_geography",
        "high_school_world_history",
        "miscellaneous",
        "professional_law",
    ]

    def __init__(self, subjects=None):
        self.subjects = subjects or self.DEFAULT_SUBJECTS

    def load(self, n=30):
        per_subject = max(1, math.ceil(n / len(self.subjects)))
        questions = []
        for subject in self.subjects:
            ds = load_dataset(
                "edinburgh-dawg/mmlu-redux-2.0",
                subject,
                split="test")
            ds = ds.select(range(min(per_subject, len(ds))))
            for row in ds:
                choices = "\n".join(
                    f"{chr(65 + i)}. {c}"
                    for i, c in enumerate(row["choices"])
                )
                q = (
                    f"{row['question']}\n{choices}\n"
                    "Answer with just the letter."
                )
                questions.append(
                    {"question": q, "gold": chr(65 + row["answer"])})
        return questions[:n]

    def grade(self, output_text, gold):
        # Take the LAST standalone letter: a model may name other options
        # before settling on its real pick.
        matches = re.findall(r"\b([A-D])\b", output_text.strip())
        if not matches:
            return False
        return matches[-1] == gold


class IFEvalBenchmark(Benchmark):
    """Easy / Instruction-following — IFEval (programmatically verifiable
    output constraints).

    Tier: "easy-to-medium" prior. No official LFM2.5-2.6B IFEval score exists;
    Liquid AI publishes strong numbers on a different in-house suite
    (IFStruct 85.49, Multi-IF 80.07, IFBench 59.17), which is suggestive of the
    general capability but is not the same benchmark.

    Simplification: the official grader implements ~25 verifiable-instruction
    checkers (instructions_registry.py in google-research/
    instruction_following_eval). This loader keeps only questions that use the
    subset implemented below — keyword existence/frequency, forbidden words,
    word/sentence/paragraph counts, lowercase, bullet-list count. Swap in the
    official registry for publication-grade rigor.
    """

    name = "IFEval"
    difficulty_tier = "easy"
    task_type = "instruction_following"

    _SUPPORTED_PREFIXES = (
        "keywords:existence",
        "keywords:frequency",
        "keywords:forbidden_words",
        "length_constraints:number_words",
        "length_constraints:number_sentences",
        "length_constraints:number_paragraphs",
        "change_case:english_lowercase",
        "detectable_format:number_bullet_lists",
    )

    def load(self, n=30):
        ds = load_dataset("google/IFEval", split="train")
        questions = []
        for row in ds:
            ids = row["instruction_id_list"]
            if not any(i.startswith(self._SUPPORTED_PREFIXES) for i in ids):
                continue
            questions.append({
                "question": row["prompt"],
                "gold": {"instruction_id_list": ids, "kwargs": row["kwargs"]},
            })
            if len(questions) >= n:
                break
        return questions

    def _check_one(self, text, instr_id, kwargs):
        try:
            if instr_id.startswith("keywords:existence"):
                return all(kw.lower() in text.lower()
                           for kw in kwargs.get("keywords", []))
            if instr_id.startswith("keywords:frequency"):
                kw = kwargs.get("keyword", "")
                target = kwargs.get("frequency", 0)
                rel = kwargs.get("relation", "at least")
                count = text.lower().count(kw.lower())
                if rel == "at least":
                    return count >= target
                return count <= target
            if instr_id.startswith("keywords:forbidden_words"):
                return all(kw.lower() not in text.lower()
                           for kw in kwargs.get("forbidden_words", []))
            if instr_id.startswith("length_constraints:number_words"):
                n_words = len(text.split())
                target = kwargs.get("num_words", 0)
                rel = kwargs.get("relation", "at least")
                if rel == "at least":
                    return n_words >= target
                return n_words <= target
            if instr_id.startswith("length_constraints:number_sentences"):
                n_sent = len(re.findall(r"[.!?]+", text))
                target = kwargs.get("num_sentences", 0)
                rel = kwargs.get("relation", "at least")
                if rel == "at least":
                    return n_sent >= target
                return n_sent <= target
            if instr_id.startswith("length_constraints:number_paragraphs"):
                n_para = len([p for p in text.split("\n\n") if p.strip()])
                return n_para == kwargs.get("num_paragraphs", 0)
            if instr_id.startswith("change_case:english_lowercase"):
                return text == text.lower()
            if instr_id.startswith("detectable_format:number_bullet_lists"):
                n_bullets = len(
                    re.findall(
                        r"^\s*[\*\-]\s",
                        text,
                        flags=re.MULTILINE))
                return n_bullets == kwargs.get("num_bullets", 0)
        except Exception:
            return False
        return True  # unsupported id slipped through -> don't penalize

    def grade(self, output_text, gold):
        ids = gold["instruction_id_list"]
        kwargs_list = gold["kwargs"]
        results = []
        for instr_id, kwargs in zip(ids, kwargs_list):
            if not instr_id.startswith(self._SUPPORTED_PREFIXES):
                continue
            results.append(
                self._check_one(
                    output_text,
                    instr_id,
                    kwargs or {}))
        return len(results) > 0 and all(results)


# ================================================================ medium tier
class MATHFullBenchmark(Benchmark):
    """Medium / Math — full MATH test set.

    Tier: "medium" is a literature-based hypothesis — MATH sits above GSM8K and
    below AIME in published difficulty orderings. No official LFM2.5-2.6B score
    exists.

    Dataset: `hendrycks/competition_math` was disabled on the Hub after a DMCA
    takedown. `DigitalLearningGmbH/MATH-lighteval` is a re-upload with the same
    schema (problem/level/type/solution) and a `default` config holding all
    subjects in one 5000-example test split.
    """

    name = "MATH-full"
    difficulty_tier = "medium"
    task_type = "math"

    def load(self, n=30):
        ds = load_dataset(
            "DigitalLearningGmbH/MATH-lighteval",
            "default",
            split="test",
        )
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for row in ds:
            gold = _extract_boxed(row["solution"])
            if gold is None:
                continue
            questions.append({"question": row["problem"], "gold": gold})
        return questions

    def grade(self, output_text, gold):
        pred = _extract_boxed(output_text)
        if pred is None:
            lines = output_text.strip().splitlines()
            # No \boxed{} and no text at all (e.g. a truncated call with no
            # final answer) — nothing to grade against; structurally incorrect.
            pred = lines[-1].strip() if lines else ""
        return pred.replace(" ", "") == gold.replace(" ", "")


class MMLUProBenchmark(Benchmark):
    """Medium / General knowledge — MMLU-Pro (10-option, reasoning-heavier MMLU).

    Tier: "medium" is a hypothesis; no official LFM2.5-2.6B score exists.

    Dataset: the TIGER-Lab/MMLU-Pro answer column is an *integer index* (0-9),
    not a letter; grade() converts it before comparing.
    """

    name = "MMLU-Pro"
    difficulty_tier = "medium"
    task_type = "general_knowledge"

    def load(self, n=30):
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test").select(range(n))
        questions = []
        for row in ds:
            options = row["options"]
            choices = "\n".join(
                f"{chr(65 + i)}. {c}" for i, c in enumerate(options))
            q = f"{row['question']}\n{choices}\nAnswer with just the letter."
            questions.append({"question": q, "gold": row["answer"]})
        return questions

    def grade(self, output_text, gold):
        # Last lettered mention; also skips the pronoun "I" in "I pick H".
        matches = re.findall(r"\b([A-J])\b", output_text.strip())
        if not matches:
            return False
        # gold is int 0-9 from the dataset; compare to the letter
        try:
            expected_letter = chr(65 + int(gold))
        except (TypeError, ValueError):
            expected_letter = str(gold)
        return matches[-1] == expected_letter


class BFCLv3Benchmark(Benchmark):
    """Medium / Agentic tool use — BFCL "simple" single-turn Python category.

    Tier: "medium" is a structural guess — one unambiguous function call is
    less demanding than multi-turn or parallel tool use. Liquid AI's published
    BFCLv4 = 56.88 covers the full multi-category suite (multi-turn, parallel,
    irrelevance detection), not this easier subset, so it is not a valid
    expectation here.

    Dataset: the gorilla repo moved to BFCL v4 and split "simple" by language,
    so this loads BFCL_v4_simple_python.json — same id/question/function schema
    and ground_truth format as the old v3 file, but not a byte-identical
    question set.

    Simplification: the official grader (`bfcl.eval_checker`) does semantic AST
    matching against a possible-answer set with type/value constraints. This is
    a lighter check — parse the call, match the function name, and require every
    required argument's value to be among the acceptable values.
    """

    name = "BFCL-v3-simple"
    difficulty_tier = "medium"
    task_type = "agentic_tool_use"

    REPO_DIR = "/kaggle/working/gorilla"
    QUESTION_GLOB = "**/BFCL_v4_simple_python.json"
    ANSWER_GLOB = "**/possible_answer/BFCL_v4_simple_python.json"

    def _ensure_repo(self):
        if not os.path.exists(self.REPO_DIR):
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/ShishirPatil/gorilla.git", self.REPO_DIR],
                check=False,
            )

    def load(self, n=30):
        self._ensure_repo()
        q_files = glob.glob(
            os.path.join(
                self.REPO_DIR,
                self.QUESTION_GLOB),
            recursive=True)
        a_files = glob.glob(
            os.path.join(
                self.REPO_DIR,
                self.ANSWER_GLOB),
            recursive=True)
        if not q_files or not a_files:
            raise FileNotFoundError(
                "BFCL_v3_simple.json / possible_answer file not found under "
                f"{self.REPO_DIR} — the gorilla repo layout may have changed; "
                "update QUESTION_GLOB/ANSWER_GLOB."
            )

        def read_jsonl(path):
            with open(path) as f:
                return [json.loads(line) for line in f if line.strip()]

        questions_raw = read_jsonl(q_files[0])[:n]
        answers_raw = {row["id"]: row["ground_truth"]
                       for row in read_jsonl(a_files[0])}

        questions = []
        for row in questions_raw:
            qid = row["id"]
            if qid not in answers_raw:
                continue
            turns = row["question"][0] if isinstance(
                row["question"][0], list) else row["question"]
            user_msg = next(
                (t["content"] for t in turns if t["role"] == "user"),
                turns[0]["content"])
            funcs_desc = json.dumps(row["function"], indent=2)
            prompt = (
                f"You can call the following function(s):\n{funcs_desc}\n\n"
                f"User request: {user_msg}\n\n"
                "Respond with exactly one Python-style function call, "
                "e.g. func_name(arg1=value1, arg2=value2)."
            )
            questions.append({"question": prompt, "gold": answers_raw[qid]})
        return questions

    def grade(self, output_text, gold):
        # gold: list like [{"func_name": {"arg": ["acceptable", "values"]}}]
        #
        # Scan call candidates from the END backwards, accepting the first
        # that parses as Python and matches a known function name with
        # acceptable argument values — defensive against scratch calls left in
        # the text ("I'll call get_x(...) actually no, get_y(...)").
        text = output_text.strip()
        candidates = list(re.finditer(r"(\w+)\s*\((.*?)\)", text, flags=re.DOTALL))
        if not candidates:
            # fallback: old greedy single-span match, for calls containing
            # literal nested parentheses inside a string argument.
            m = re.search(r"(\w+)\s*\((.*)\)", text, flags=re.DOTALL)
            candidates = [m] if m else []

        for match in reversed(candidates):
            called_name, args_str = match.group(1), match.group(2)
            gold_entry = next(
                (entry[called_name] for entry in gold if called_name in entry),
                None)
            if gold_entry is None:
                continue
            try:
                call_ast = ast.parse(f"f({args_str})").body[0].value
                called_kwargs = {
                    kw.arg: ast.literal_eval(kw.value)
                    for kw in call_ast.keywords}
            except Exception:
                continue
            ok = all(
                arg_name in called_kwargs
                and str(called_kwargs[arg_name]) in [str(v) for v in acceptable]
                for arg_name, acceptable in gold_entry.items()
            )
            if ok:
                return True
        return False


# ================================================================== hard tier
class AIME2025Benchmark(Benchmark):
    """Hard / Math — AIME 2025 (competition problems, integer answers).

    Tier: "hard", anchored to a real number — Liquid AI publishes AIME25 =
    51.87 for LFM2.5-2.6B ("Deploy Agents Everywhere" blog post / HF model
    card). AIME 2025 rather than 2024 lowers contamination risk.
    """

    name = "AIME2025"
    difficulty_tier = "hard"
    task_type = "math"

    def load(self, n=30):
        # NOTE: HuggingFaceH4/aime_2025 no longer resolves on the Hub.
        # test-time-compute/aime_2025 is the current replacement: single
        # "test" split, 30 problems (AIME 2025 I + II combined), fields
        # question/answer instead of problem/answer.
        ds = load_dataset("test-time-compute/aime_2025", split="test")
        ds = ds.select(range(min(n, 30, len(ds))))
        question_key = "question" if "question" in ds.column_names else "problem"
        questions = []
        for row in ds:
            questions.append({
                "question": row[question_key],
                "gold": str(row["answer"]).strip(),
            })
        return questions

    def grade(self, output_text, gold):
        numbers = re.findall(r"-?\d+", output_text)
        if not numbers:
            return False
        return numbers[-1].strip() == gold


class GPQADiamondBenchmark(Benchmark):
    """Hard / PhD-level science — GPQA-Diamond.

    Tier: "hard" is a literature-based hypothesis — PhD-level and adversarially
    filtered against search. No official LFM2.5-2.6B score exists.

    Access: gated on HF — accept the terms at
    huggingface.co/datasets/Idavidrein/gpqa and set HF_TOKEN before .load().

    Loading: the raw file always lists the correct answer first, so options are
    shuffled per question with a fixed per-question seed — removes the
    positional artifact while keeping the question set reproducible.
    """

    name = "GPQA-Diamond"
    difficulty_tier = "hard"
    task_type = "phd_science"

    def load(self, n=30):
        ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for i, row in enumerate(ds):
            options = [
                row["Correct Answer"],
                row["Incorrect Answer 1"],
                row["Incorrect Answer 2"],
                row["Incorrect Answer 3"],
            ]
            rng = random.Random(i)  # deterministic per-question shuffle
            order = list(range(4))
            rng.shuffle(order)
            shuffled = [options[j] for j in order]
            gold_letter = chr(65 + order.index(0))
            choices = "\n".join(
                f"{chr(65 + i)}. {c}" for i, c in enumerate(shuffled))
            q = f"{row['Question']}\n{choices}\nAnswer with just the letter."
            questions.append({"question": q, "gold": gold_letter})
        return questions

    def grade(self, output_text, gold):
        # Last standalone letter, as in MMLURedux.grade() above.
        matches = re.findall(r"\b([A-D])\b", output_text.strip())
        if not matches:
            return False
        return matches[-1] == gold


class LiveCodeBenchV6Benchmark(Benchmark):
    """Hard / Coding — LiveCodeBench v6.

    Tier: "hard", anchored to a real number — Liquid AI publishes
    LiveCodeBench-v6 = 59.41 for LFM2.5-2.6B.

    Simplification: grading uses `public_test_cases` only (stdin/stdout pairs,
    run in a subprocess with a timeout). `private_test_cases` are
    zlib+base64+pickle-encoded and are not decoded here, so scores are an
    optimistic bound relative to the official public+private protocol. A
    problem counts as correct only if every public test case passes.
    """

    name = "LiveCodeBench-v6"
    difficulty_tier = "hard"
    task_type = "coding"
    TIME_LIMIT_S = 6

    def load(self, n=30):
        # NOTE: livecodebench/code_generation_lite uses a Python loading
        # script (with a version_tag= kwarg), which recent `datasets`
        # versions refuse to run ("Dataset scripts are no longer
        # supported"). lighteval/code_generation_lite is a parquet mirror
        # with the identical row schema, using ordinary config names
        # (e.g. "release_v6") instead of a version_tag kwarg.
        ds = load_dataset(
            "lighteval/code_generation_lite",
            "release_v6",
            split="test",
        )
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for row in ds:
            try:
                tests = json.loads(row["public_test_cases"])
            except Exception:
                tests = []
            if not tests:
                continue
            prompt = (
                f"{row['question_content']}\n\n"
                f"Starter code (if any):\n{row.get('starter_code', '')}\n\n"
                "Write a complete Python solution that reads input from stdin "
                "and writes output to stdout. Return the code in a single "
                "```python fenced block."
            )
            questions.append({"question": prompt, "gold": tests})
        return questions

    @staticmethod
    def _extract_code(text):
        # Take the LAST fenced block: a model may draft scratch snippets
        # before its final version.
        blocks = re.findall(r"```python\s*(.*?)```", text, flags=re.DOTALL)
        if not blocks:
            blocks = re.findall(r"```\s*(.*?)```", text, flags=re.DOTALL)
        return blocks[-1] if blocks else text

    def grade(self, output_text, gold):
        code = self._extract_code(output_text)
        for case in gold:
            try:
                with tempfile.NamedTemporaryFile(
                    "w", suffix=".py", delete=False
                ) as f:
                    f.write(code)
                    path = f.name
                result = subprocess.run(
                    ["python3", path],
                    input=case.get("input", ""),
                    capture_output=True,
                    text=True,
                    timeout=self.TIME_LIMIT_S,
                )
                if result.stdout.strip() != case.get("output", "").strip():
                    return False
            except Exception:
                return False
            finally:
                try:
                    os.remove(path)
                except OSError:
                    pass
        return True

## 5. Data collection

One row per model call, labeled with benchmark / tier / task type / compute level / sample, so
every number downstream can be traced back to the calls that produced it.

### Keeping the thinking/response boundary alive

vLLM's OpenAI-compatible server decodes with `skip_special_tokens=True` by default, and LFM2.5's
` thinking` / ` response` tags **are** special tokens — so the boundary would be stripped from
`message.content` before this code ever sees it. Every request therefore passes
`skip_special_tokens=False`, and the text is split on the **last** ` response` occurrence
(`rsplit`), which is defensive against the literal English word appearing earlier in free text.

Without this, the split silently never fires, the thinking column stays empty, and the grader
receives thinking and answer glued together.

### Separate thinking and output token counts

The OpenAI-compatible API returns **one** aggregate `completion_tokens` figure, so there is no
way to get a per-half count from the response. Once the text is split, each half is re-encoded
locally with the tokenizer from the pre-flight check.

This is an **approximation** — re-tokenizing text is not bit-identical to slicing the original
token stream, because merges at the boundary can differ by a token or two. It is close enough for
curve-shaped analysis, and it is what makes "long think, short answer" distinguishable from the
reverse. Do not read `thinking_tokens` as an exact count.

### Truncation semantics

If no marker ever appears but text was produced, generation hit `max_tokens` **mid-thought**.
That text is unfinished reasoning, not an answer, so it is stored as `thinking` with
`output = ""` and `was_truncated = True`. The grader then scores it incorrect — which is
structurally right, since the model never answered. Keeping this flag separate is what lets the
analysis distinguish "got it wrong" from "ran out of budget"; the latter is the textbook
underthinking signature.

### Operational machinery

- **Round-robin over clients** — load-balances the data-parallel servers.
- **16 workers per GPU** — the run is decode-bound and vLLM batches server-side; concurrency is
  what turns 2,700 sequential calls into ~25 minutes.
- **Checkpoint every 25 rows, written temp-file-then-rename** — Kaggle sessions get killed, and
  an atomic replace means a crash mid-write cannot corrupt the CSV. On rerun the `done` set is
  rebuilt from the checkpoint and completed calls are skipped.
- **`effective_max_tokens`** — must sit above the budget, or the model is cut off before it can
  use its answer margin; the outer `min` keeps prompt + output inside `max_model_len`.
- **Live KPIs** — `tok/s`, ETA, cost, and the marker hit rate, which acts as a canary: if it
  drops mid-run, a server restarted without the `skip_special_tokens` override.

In [66]:
import time
import pandas as pd
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed


class ThinkingBudgetEvaluator:
    """Concurrent data collector with checkpoint/resume and live KPI logs.

    One row per model call, labeled with benchmark / difficulty_tier /
    task_type / compute_level / sample. Every 25 rows the checkpoint CSV is
    flushed, so a dropped session resumes instead of restarting.

    Each request is sent with `skip_special_tokens=False` so the " thinking" /
    " response" boundary tokens survive decoding; the text is split on the last
    " response" marker, and each half is re-tokenized locally to give separate
    `thinking_tokens` / `output_tokens` columns.
    """

    def __init__(self, server, max_workers=24,
                 checkpoint="/kaggle/working/h1_raw_results.csv",
                 tokenizer=None):
        self.server = server
        self.ports = server.ports
        self.clients = [
            OpenAI(base_url=f"http://localhost:{p}/v1", api_key="dummy")
            for p in self.ports
        ]
        self._rr = 0  # round-robin across the DP servers
        self.max_workers = max_workers
        self.checkpoint = checkpoint
        self.rows = []
        self.done = set()
        # Used only to count tokens on each side of the split; defaults to
        # the `_tok` loaded by the pre-flight check.
        self.tokenizer = tokenizer if tokenizer is not None else globals().get("_tok")
        self._response_marker = globals().get("RESPONSE_MARKER")
        if self.tokenizer is None:
            raise RuntimeError(
                "No tokenizer available for thinking/output token counting — "
                "run cell 3 first (it defines the global `_tok`), or pass "
                "tokenizer=... explicitly.")
        # Read max_model_len from the vLLM config to cap API calls.
        # Reserve ~256 tokens of context for the input prompt so
        # prompt + output never exceed max_model_len (vLLM 400 otherwise).
        self.max_model_len = 32768  # safe default
        self.max_output_tokens = self.max_model_len - 256
        try:
            import yaml
            with open(os.path.join(WORK_DIR, "vllm-config.yaml")) as f:
                cfg = yaml.safe_load(f)
            self.max_model_len = int(cfg.get("max-model-len", 32768))
        except Exception:
            # No pyyaml? parse manually
            try:
                for line in open(os.path.join(WORK_DIR, "vllm-config.yaml")):
                    if line.startswith("max-model-len:"):
                        self.max_model_len = int(line.split(":")[1].strip())
            except Exception:
                pass
        print(f"[evaluator] max_model_len={self.max_model_len} "
              f"(output capped at {self.max_output_tokens})", flush=True)
        self._load_checkpoint()

    def _load_checkpoint(self):
        if os.path.exists(self.checkpoint):
            df = pd.read_csv(self.checkpoint)
            self.rows = df.to_dict("records")
            self.done = {
                (r["benchmark"], r["question_id"], r["compute_level"],
                 r["sample_index"])
                for r in self.rows
            }
            print(f"[resume] {len(self.done)} calls already in checkpoint — "
                  f"skipping them", flush=True)

    def _flush(self):
        if not self.rows:
            return
        tmp = self.checkpoint + ".tmp"
        pd.DataFrame(self.rows).to_csv(tmp, index=False)
        os.replace(tmp, self.checkpoint)

    # Decoded form of the dedicated single-token id 124902, confirmed by the
    # pre-flight tokenizer check.

    def query(self, question, budget, max_tokens, temperature=0.7):
        t0 = time.time()
        client = self.clients[self._rr % len(self.clients)]
        self._rr += 1
        response = client.chat.completions.create(
            model=self.server.model,
            messages=[{"role": "user", "content": question}],
            max_tokens=max_tokens,
            temperature=temperature,
            extra_body={
                "vllm_xargs": {"max_thinking_tokens": budget},
                # vLLM decodes with skip_special_tokens=True by default,
                # which would drop the boundary tokens the split needs.
                "skip_special_tokens": False,
            },
        )
        elapsed = time.time() - t0
        full_text = response.choices[0].message.content or ""
        tokens_used = response.usage.completion_tokens

        # With `--reasoning-parser` set, vLLM would split reasoning from
        # content server-side and expose it here. This config sets none, so
        # this is normally empty and the text split below is what runs.
        reasoning_text = getattr(
            response.choices[0].message, "reasoning_content", None) or ""

        if reasoning_text:
            thinking = str(reasoning_text).strip()
            output = full_text.strip()
            was_truncated = False
            hit_marker = True
        elif self._response_marker in full_text:
            # rsplit(..., 1): split on the LAST occurrence. The budget
            # processor should only ever force this marker once, but
            # splitting from the right protects against the literal word
            # "response" coincidentally showing up earlier in free text.
            thinking, output = full_text.split(self._response_marker, 1)            
            thinking, output = thinking.strip(), output.strip()
            was_truncated = False
            hit_marker = True
        elif full_text.strip():
            # No " response" marker at all: generation hit max_tokens before
            # the model reached a final answer. The text is unfinished
            # reasoning, so it is bucketed as thinking with no output to grade
            # — structurally incorrect, because the model never answered.
            thinking = full_text.strip()
            output = ""
            was_truncated = True
            hit_marker = False
        else:
            thinking, output = "", ""
            was_truncated = tokens_used >= max_tokens
            hit_marker = False

        # The API returns only one aggregate `completion_tokens`, so each
        # half is re-tokenized locally. Approximate (merges at the boundary can
        # shift a token or two) but enough to tell "long think, short answer"
        # from the reverse.
        thinking_tokens = (
            len(self.tokenizer.encode(thinking, add_special_tokens=False))
            if thinking else 0)
        output_tokens = (
            len(self.tokenizer.encode(output, add_special_tokens=False))
            if output else 0)

        return (output, thinking, tokens_used, thinking_tokens,
                output_tokens, hit_marker, elapsed, was_truncated)

    def _call_one(self, q, bench, qi, budget, sample_idx,
                  max_tokens, temperature, answer_margin):
        # Clamp to server's max_model_len (minus input headroom) to avoid 400s
        effective_max_tokens = min(
            max(max_tokens, budget + answer_margin),
            self.max_output_tokens)
        (output, thinking, tokens, thinking_tokens, output_tokens,
         hit_marker, elapsed, was_truncated) = self.query(
            q["question"], budget, effective_max_tokens, temperature)
        return {
            "question_id": f"{bench.name}_{qi}",
            "benchmark": bench.name,
            "difficulty_tier": bench.difficulty_tier,
            "task_type": bench.task_type,
            "compute_level": budget,
            "sample_index": sample_idx,
            "raw_model_output": output,
            "thinking_text": thinking,
            "gold": q["gold"],
            "is_correct": bench.grade(output, q["gold"]),
            "tokens_used": tokens,
            "thinking_tokens": thinking_tokens,
            "output_tokens": output_tokens,
            "hit_response_marker": hit_marker,
            "latency_s": elapsed,
            "was_truncated": was_truncated,
            "hit_marker_empty_output": hit_marker and not output.strip(),
        }

    def collect(self, benchmarks, compute_levels, n_questions=30, n_samples=5,
                max_tokens=2000, temperature=0.7, answer_margin=800):
        all_questions = {b.name: b.load(n_questions) for b in benchmarks}
        for b in benchmarks:
            print(f"[data] {b.name} -> {len(all_questions[b.name])} questions",
                  flush=True)

        tasks = []
        for b in benchmarks:
            for qi, q in enumerate(all_questions[b.name]):
                for budget in compute_levels:
                    for si in range(n_samples):
                        key = (b.name, f"{b.name}_{qi}", budget, si)
                        if key in self.done:
                            continue
                        tasks.append((q, b, qi, budget, si))

        total = len(tasks)
        if total == 0:
            print("[collect] nothing to do — all calls already done", flush=True)
            return pd.DataFrame(self.rows)
        print(f"[collect] {total} pending calls, {self.max_workers} workers",
              flush=True)

        started = time.time()
        tokens = 0
        done = 0
        with ThreadPoolExecutor(max_workers=self.max_workers) as pool:
            futures = {
                pool.submit(self._call_one, *t, max_tokens, temperature,
                            answer_margin): t
                for t in tasks
            }
            for fut in as_completed(futures):
                row = fut.result()
                self.rows.append(row)
                tokens += row["tokens_used"]
                done += 1
                since = time.time() - started
                if done % 25 == 0 or done == total:
                    speed = tokens / max(since, 1e-3)
                    eta = (total - done) / (done / since) if done else 0
                    print(
                        f"[collect] {done}/{total} ({done*100//total}%) "
                        f"tok/s={speed:.0f} ETA={eta/60:.0f}min "
                        f"elapsed={since/60:.1f}min ok={row['is_correct']} "
                        f"marker_hit_rate={sum(r['hit_response_marker'] for r in self.rows)/len(self.rows):.2f} "
                        f"est_cost=~${since*GPU_RATE_USD_S:.2f}",
                        flush=True)
                    self._flush()
        self._flush()
        print(f"[collect] DONE: {len(self.rows)} rows in {self.checkpoint}",
              flush=True)
        return pd.DataFrame(self.rows)

## 6. Pilot

Two far-apart budgets, one question, one sample per benchmark — a cheap confirmation that the
budget knob actually changes behaviour *before* spending 2,700 calls. Three things must hold:

1. **`marker_hit_rate` ≈ 1.0** — the thinking/response split is working. A low rate means
   `skip_special_tokens` did not take effect, the tag string is wrong, or budgets are too tight
   relative to `answer_margin`.
2. **`thinking_tokens` grows with budget** — the cap is binding.
3. **`output_tokens` stays flat** — the budget is being spent on reasoning, not leaking into a
   longer final answer.

If any of these fails, stop and fix it here. This is the cheapest available guard against a full
run of plausible-looking garbage.

In [67]:
# Set True to wipe accumulated results and start fresh with the current
# benchmarks/COMPUTE_LEVELS. Leave False to resume normally.
RESET_CHECKPOINTS = True

if RESET_CHECKPOINTS:
    for _f in ["/kaggle/working/h1_raw_results.csv",
               "/kaggle/working/h1_pilot_results.csv"]:
        if os.path.exists(_f):
            os.remove(_f)
            print(f"[reset] removed {_f}")
    print("[reset] checkpoints cleared — next run starts from zero")
else:
    print("[reset] RESET_CHECKPOINTS=False — resuming from any existing checkpoint")

[reset] removed /kaggle/working/h1_pilot_results.csv
[reset] checkpoints cleared — next run starts from zero


In [68]:
benchmarks = [
    GSM8KBenchmark(),            # easy / math
    MMLURedux(),                 # easy / general_knowledge
    IFEvalBenchmark(),           # easy / instruction_following
    MATHFullBenchmark(),         # medium / math
    MMLUProBenchmark(),          # medium / general_knowledge
    BFCLv3Benchmark(),           # medium / agentic_tool_use
    AIME2025Benchmark(),         # hard / math
    GPQADiamondBenchmark(),      # hard / phd_science
    LiveCodeBenchV6Benchmark(),  # hard / coding
]

PILOT_LEVELS = [0,256, 8192]

evaluator = ThinkingBudgetEvaluator(server, max_workers=MAX_WORKERS,
                                    checkpoint="/kaggle/working/h1_pilot_results.csv")
pilot_df = evaluator.collect(
    benchmarks=benchmarks, compute_levels=PILOT_LEVELS,
    n_questions=1, n_samples=1,
)
pilot_df.to_csv("/kaggle/working/h1_pilot_results.csv", index=False)

print("\n" + "=" * 60)
print("[pilot] SANITY CHECKS")
print("=" * 60)

seen = pilot_df.groupby(["benchmark", "difficulty_tier", "task_type"]).size()
missing = {(b.name, b.difficulty_tier, b.task_type) for b in benchmarks} - set(seen.index)
print(f"[pilot] missing benchmarks: {missing if missing else 'none'}")

# Parser sanity: below ~0.9 the thinking/response split is not working and the
# full sweep should not be started.
marker_rate = pilot_df["hit_response_marker"].mean()
print(f"\n[pilot] response-marker hit rate: {marker_rate:.2%} "
      f"({'OK' if marker_rate > 0.9 else 'INVESTIGATE — parsing may still be broken'})")

budget_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["tokens_used"]
    .mean().unstack("compute_level"))
print("\n[pilot] mean completion tokens (total) by compute_level (should rise left->right):")
print(budget_effect)

# thinking_tokens should track compute_level closely; output_tokens should stay
# comparatively flat — a final answer has no reason to grow with the budget.
think_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["thinking_tokens"]
    .mean().unstack("compute_level"))
out_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["output_tokens"]
    .mean().unstack("compute_level"))
print("\n[pilot] mean thinking_tokens by compute_level:")
print(think_effect)
print("\n[pilot] mean output_tokens by compute_level:")
print(out_effect)

bench_acc = pilot_df.groupby(["benchmark", "difficulty_tier"])["is_correct"].mean()
print("\n[pilot] accuracy by benchmark:")
print(bench_acc)

[evaluator] max_model_len=16384 (output capped at 32512)
[data] GSM8K -> 1 questions
[data] MMLU-Redux -> 1 questions
[data] IFEval -> 1 questions
[data] MATH-full -> 1 questions
[data] MMLU-Pro -> 1 questions
[data] BFCL-v3-simple -> 1 questions
[data] AIME2025 -> 1 questions
[data] GPQA-Diamond -> 1 questions
[data] LiveCodeBench-v6 -> 1 questions
[collect] 27 pending calls, 32 workers
[collect] 25/27 (92%) tok/s=245 ETA=0min elapsed=1.8min ok=True marker_hit_rate=1.00 est_cost=~$0.03
[collect] 27/27 (100%) tok/s=164 ETA=0min elapsed=3.9min ok=False marker_hit_rate=1.00 est_cost=~$0.08
[collect] DONE: 27 rows in /kaggle/working/h1_pilot_results.csv

[pilot] SANITY CHECKS
[pilot] missing benchmarks: none

[pilot] response-marker hit rate: 100.00% (OK)

[pilot] mean completion tokens (total) by compute_level (should rise left->right):
compute_level       0       256     8192
benchmark                               
AIME2025          1917.0  1629.0  2880.0
BFCL-v3-simple      19.0   174

## 7. Full sweep

The 2,700-call run. Roughly 20–30 minutes on 2×T4, comfortably inside the free weekly quota.

Re-running this cell after a dropped session **resumes** rather than restarting — completed
(benchmark, question, budget, sample) combinations are skipped.

In [ ]:
# Re-running this cell after a dropped session resumes from the checkpoint CSV.

COMPUTE_LEVELS = [0, 256, 512, 1024, 2048, 4096, 8192]
N_QUESTIONS = 10
N_SAMPLES = 5

t_all = time.time()
evaluator = ThinkingBudgetEvaluator(server, max_workers=MAX_WORKERS,
                                    checkpoint="/kaggle/working/h1_raw_results.csv")
raw_df = evaluator.collect(
    benchmarks=benchmarks,
    compute_levels=COMPUTE_LEVELS,
    n_questions=N_QUESTIONS,
    n_samples=N_SAMPLES,
)
# Same empty-string -> NaN round-trip issue as the pilot cell — fix before saving.
raw_df[["raw_model_output", "thinking_text"]] = (
    raw_df[["raw_model_output", "thinking_text"]].fillna(""))

raw_df.to_csv("/kaggle/working/h1_raw_results.csv", index=False)

elapsed_s = time.time() - t_all
total_tok = int(raw_df["tokens_used"].sum())
print("\n" + "=" * 60)
print("[run] FULL SWEEP COMPLETE")
print(f"      rows={len(raw_df)}  tokens={total_tok}")
print(f"      wall={elapsed_s/60:.1f} min  avg={total_tok/max(elapsed_s,1):.0f} tok/s")
print(f"      overall acc={raw_df['is_correct'].mean():.3f}")
# A mid-run drop in the marker hit rate flags e.g. a server restart losing the
# skip_special_tokens override, or prompts pushing budgets past max_model_len.
print(f"      response-marker hit rate={raw_df['hit_response_marker'].mean():.2%}")
print(f"      mean thinking_tokens={raw_df['thinking_tokens'].mean():.0f}  "
      f"mean output_tokens={raw_df['output_tokens'].mean():.0f}")
print(f"      est cost ~${elapsed_se33*GPU_RATE_USD_S:.2f} (Kaggle quota covers this)")
print("=" * 60)
raw_df.head()

[evaluator] max_model_len=16384 (output capped at 32512)
[data] GSM8K -> 10 questions
[data] MMLU-Redux -> 10 questions
[data] IFEval -> 10 questions
[data] MATH-full -> 10 questions
[data] MMLU-Pro -> 10 questions
[data] BFCL-v3-simple -> 10 questions
[data] AIME2025 -> 10 questions
[data] GPQA-Diamond -> 10 questions
[data] LiveCodeBench-v6 -> 10 questions
[collect] 3150 pending calls, 32 workers
[collect] 25/3150 (0%) tok/s=381 ETA=53min elapsed=0.4min ok=True marker_hit_rate=1.00 est_cost=~$0.01
